<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 vsep - Audio Stem Separator Demo

Welcome to the **vsep** interactive demo! This notebook lets you separate audio into different stems (vocals, drums, bass, instruments) using state-of-the-art AI models.

## Features
- ⚡ **Fast Processing** - Optimized downloads and inference
- 🎯 **Multiple Models** - Choose from 100+ UVR models
- 🎚️ **Easy to Use** - Simple interface for audio separation
- 💾 **Download Results** - Get your separated stems

---

## 1️⃣ Setup

First, let's install vsep and its dependencies. This takes about 2-3 minutes.

In [ ]:
#@title Install vsep
#@markdown Run this cell to install vsep (first time only)

# Clone the repository
!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep

# Install dependencies
!uv pio install -r requirements.txt

import os
os.chdir('/content/vsep')

import torch
if torch.cuda.is_available():
    print(f"✅ NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected, will use CPU (slower)")
    print("💡 Tip: Go to Runtime > Change runtime type > Hardware accelerator > GPU")

## 2️⃣ Upload Your Audio

Upload the audio file you want to separate. Supported formats: MP3, WAV, FLAC, OGG, M4A

In [ ]:
#@title Upload Audio File
#@markdown Upload your audio file for separation

from google.colab import files
import os

print("📁 Please upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {audio_file}")

    # Create output directory
    os.makedirs("output", exist_ok=True)
    os.makedirs("models", exist_ok=True)
else:
    print("❌ No file uploaded. Please run this cell again.")

## 3️⃣ Choose Model

Select the model you want to use for separation. Different models work better for different types of music.

In [ ]:
#@title Select Model
#@markdown Choose a separation model

#@markdown ### Recommended Models:
model_choice = "ht-demucs_ft.yaml" #@param ["ht-demucs_ft.yaml", "BS-Roformer-Viperx-1297.ckpt", "UVR-MDX-NET-Inst_1.onnx", "Mel-Roformer-Viperx-1053.ckpt", "Kim_Vocal_2.onnx"]

model_descriptions = {
    "ht-demucs_ft.yaml": "🎸 Demucs v4 - Best all-rounder (4 stems: vocals, drums, bass, other)",
    "BS-Roformer-Viperx-1297.ckpt": "🎤 BS-Roformer - Excellent vocal quality (2 stems)",
    "UVR-MDX-NET-Inst_1.onnx": "🎹 MDX-Net - Good instrumental/vocal split (2 stems)",
    "Mel-Roformer-Viperx-1053.ckpt": "🎵 Mel-Roformer - High-quality vocal separation (2 stems)",
    "Kim_Vocal_2.onnx": "🎤 Kim Vocal - Specialized for vocal extraction (2 stems)"
}

print(f"\n📊 Selected Model: {model_choice}")
print(f"📝 Description: {model_descriptions.get(model_choice, 'Custom model')}")

# Store the choice
selected_model = model_choice

## 4️⃣ Separate Audio

Now let's separate your audio file using the selected model.

In [ ]:
#@title Run Separation
#@markdown Click to start the separation process

# Import from the correct location
from vsep import Separator
import vsep.config.variables as cfg
import os
from IPython.display import Audio, display
import glob

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("🚀 Starting separation process...")
print(f"📁 Input file: {audio_file}")
print(f"🎯 Model: {selected_model}")
print("\n⏳ This may take 2-5 minutes depending on the model and audio length...\n")

# Initialize separator
separator = Separator(
    model_file_dir="./models",
    output_dir="./output",
    use_soundfile=True,
)

# Run separation
output_files = separator.separate(audio_file, selected_model)

print(f"\n✅ Separation complete!")
print(f"📂 Output files: {len(output_files)}")
for f in output_files:
    print(f"   - {f}")

## 5️⃣ Listen to Results

Play back the separated stems to hear the results.

In [ ]:
#@title Listen to Separated Stems
#@markdown Browse and play the separated audio files

import os
from IPython.display import Audio
import glob

# Get all output files
output_files = sorted(glob.glob("output/*.wav"))

if output_files:
    print(f"🎵 Found {len(output_files)} separated stems:\n")

    for i, file_path in enumerate(output_files, 1):
        filename = os.path.basename(file_path)
        print(f"{i}. {filename}")
        display(Audio(file_path))
        print("-" * 50)
else:
    print("❌ No output files found. Please run the separation first.")

## 6️⃣ Download Results

Download the separated stems to your computer.

In [ ]:
#@title Download Separated Stems
#@markdown Download all separated audio files

from google.colab import files
import glob
import os

output_files = glob.glob("output/*.wav")

if output_files:
    print(f"📥 Downloading {len(output_files)} files...")
    for file_path in output_files:
        files.download(file_path)
        print(f"✅ Downloaded: {os.path.basename(file_path)}")
else:
    print("❌ No files to download. Please run separation first.")

## 📚 Additional Information

### Model Recommendations

- **For Vocals:** Use `BS-Roformer-Viperx-1297.ckpt` or `Mel-Roformer-Viperx-1053.ckpt`
- **For Instruments:** Use `ht-demucs_ft.yaml` (gives you all 4 stems)
- **For Karaoke:** Use `UVR-MDX-NET-Inst_1.onnx`
- **Best Quality:** Try ensemble mode with multiple models

### Tips

1. **GPU Acceleration:** Make sure you're using a GPU runtime for faster processing
   - Go to `Runtime > Change runtime type > Hardware accelerator > GPU`
2. **Long Files:** For files > 10 minutes, consider splitting them first
3. **Quality:** Higher quality models take longer but produce better results
4. **Memory:** If you run out of memory, try a smaller model or shorter audio

### Troubleshooting

**"No GPU detected"**
- Go to Runtime > Change runtime type > GPU
- Restart the runtime if needed

**"Download failed"**
- The model download might have timed out
- Re-run the separation cell (downloads are cached)

**"Out of memory"**
- Try a shorter audio file
- Use a simpler model (MDX-Net instead of Roformer)

**"Module not found"**
- Make sure you ran the installation cell first
- Try restarting the runtime (Runtime > Restart runtime)

---

### Links

- 📦 **GitHub:** [https://github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- 📖 **Documentation:** [https://github.com/BF667-IDLE/vsep#readme](https://github.com/BF667-IDLE/vsep#readme)
- 🐛 **Issues:** [https://github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)
- 📦 **Install Guide:** [INSTALL.md](https://github.com/BF667-IDLE/vsep/blob/main/INSTALL.md)

**Made with ❤️ using vsep**